# 06 - Train YOLO

Stage T. Trains from `configs/experiments/train_yolo_v1.yaml` on the newest
dataset run (or a pinned one). Outputs to `models/<run_id>/` with run.json.

Speed note: training straight off Drive is I/O-bound - `STAGE_TO_LOCAL = True`
copies the dataset to the Colab VM disk first (worth it above ~1k images).
**GPU runtime required.**

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None   # set to redirect all artifact I/O (e.g. for tests)
RAW_ROOT_OVERRIDE = None    # set to point at a different raw-sequence root

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")

In [ ]:
import json as _json
import shutil

import yaml

from vision_uss_research.datasets.yolo_export import materialize_yolo_dataset
from vision_uss_research.runs import finish_run, new_run_id, start_run

cfg = yaml.safe_load((REPO_DIR / "configs" / "experiments" / "train_yolo_v1.yaml").read_text())
cfg.update({
    # per-session overrides, e.g.: "epochs": 5,
})
DATASET_RUN = None        # run id under datasets/; None = newest
STAGE_TO_LOCAL = IN_COLAB

dataset_dirs = sorted((ARTIFACTS / "datasets").glob("*"), reverse=True)
dataset_dir = (ARTIFACTS / "datasets" / DATASET_RUN) if DATASET_RUN else next(
    (d for d in dataset_dirs if (d / "dataset_manifest.csv").exists()
     or (d / "dataset.yaml").exists()), None)
assert dataset_dir is not None, "no dataset run found - run notebook 05 first"

manifest = dataset_dir / "dataset_manifest.csv"
if manifest.exists():
    # manifest dataset: images live only in the frame cache; materialize them
    # onto local disk (sharded per sequence). Progress heartbeats to the run
    # dir on Drive -> notebook 05's status cell shows it across disconnects.
    prov = _json.loads((dataset_dir / "provenance.json").read_text())
    local_dir = (Path("/content/dataset") / dataset_dir.name if IN_COLAB
                 else ARTIFACTS / "datasets_local" / dataset_dir.name)
    result = materialize_yolo_dataset(
        manifest,
        {k: Path(v) for k, v in prov["frames_roots"].items()},
        local_dir, class_names=prov["class_names"],
        heartbeat_file=dataset_dir / "progress.json")
    print(f"materialized: {result['images_written']} written, "
          f"{result['images_skipped_existing']} resumed, "
          f"{len(result['missing_frames'])} missing")
    train_dataset_dir = local_dir
else:
    # legacy image-based dataset dirs
    if STAGE_TO_LOCAL:
        local_dir = Path("/content/dataset") / dataset_dir.name
        if not (local_dir / "dataset.yaml").exists():
            print(f"staging dataset to local disk: {local_dir} ...")
            shutil.copytree(dataset_dir, local_dir, dirs_exist_ok=True)
        train_dataset_dir = local_dir
    else:
        train_dataset_dir = dataset_dir
    data_yaml = train_dataset_dir / "dataset.yaml"
    data_cfg = yaml.safe_load(data_yaml.read_text())
    data_cfg["path"] = str(train_dataset_dir.resolve())
    data_yaml.write_text(yaml.safe_dump(data_cfg))

RUN_ID = new_run_id(cfg["stage"], cfg["tag"])
RUN_DIR = ARTIFACTS / "models" / RUN_ID
start_run(RUN_DIR, config=cfg, inputs=[f"datasets/{dataset_dir.name}"],
          repo_root=REPO_DIR)
print(f"dataset : {dataset_dir.name}\nmodel run: {RUN_DIR}")

In [ ]:
from ultralytics import YOLO

model = YOLO(cfg["model"])
results = model.train(
    data=str(train_dataset_dir / "dataset.yaml"),
    epochs=cfg["epochs"], imgsz=cfg["imgsz"], batch=cfg["batch"],
    seed=cfg["seed"],
    mosaic=cfg["augment"]["mosaic"], copy_paste=cfg["augment"]["copy_paste"],
    fliplr=cfg["augment"]["fliplr"],
    project=str(RUN_DIR), name="train", exist_ok=True,
)
BEST_WEIGHTS = RUN_DIR / "train" / "weights" / "best.pt"
print(f"best weights: {BEST_WEIGHTS}")

In [ ]:
# validation metrics on the val split (test stays untouched until notebook 07)
val_metrics = YOLO(str(BEST_WEIGHTS)).val(
    data=str(train_dataset_dir / "dataset.yaml"), split="val")
summary = {
    "mAP50": round(float(val_metrics.box.map50), 4),
    "mAP50_95": round(float(val_metrics.box.map), 4),
    "per_class_mAP50": {name: round(float(m), 4) for name, m in
                        zip(val_metrics.names.values(), val_metrics.box.ap50)},
    "best_weights": str(BEST_WEIGHTS),
}
print(_json.dumps(summary, indent=1))
finish_run(RUN_DIR, summary=summary)
print("next: notebook 07 (post-training eval on the gold test split)")